# 00 - Connection Test

**Purpose:** Verify that your Azure credentials are correctly configured before running other notebooks.

**What this does:**
1. Loads your `.env` file
2. Checks that all required environment variables are set
3. Makes a small test call to each Azure service
4. Reports which services are ready

> **Just click `Run All` (or `Shift+Enter` on each cell) - no changes needed!**

## Step 1: Load Environment Variables

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Load .env file from the project root
load_dotenv(os.path.join("..", ".env"))

print("Environment variables loaded!")

## Step 2: Run Environment Check

This checks whether all required Azure keys and endpoints are set in your `.env` file.

In [ ]:
# Run the environment checker
sys.path.insert(0, os.path.join("..", "scripts"))
from check_env import main as check_env_main

result = check_env_main()
if result == 0:
    print("\n All checks passed! You can proceed to the other notebooks.")
else:
    print("\n Some services are not configured. See the messages above.")
    print("Open the .env file in the project root and add the missing values.")
    print("Refer to: docs/01_azure_portal_setup.md")

## Step 3: Test Azure AI Services (Vision, Language, Content Safety)

This makes a quick API call to verify your Azure AI Services multi-service endpoint works.

In [ ]:
import requests

endpoint = os.environ.get("AZURE_AI_ENDPOINT", "").rstrip("/")
key = os.environ.get("AZURE_AI_KEY", "")

if endpoint and key and "YOUR" not in endpoint:
    # Test with the Language service - detect language of a simple sentence
    url = f"{endpoint}/language/:analyze-text?api-version=2023-04-01"
    headers = {"Ocp-Apim-Subscription-Key": key, "Content-Type": "application/json"}
    body = {
        "kind": "LanguageDetection",
        "parameters": {"modelVersion": "latest"},
        "analysisInput": {"documents": [{"id": "1", "text": "Hello world"}]}
    }
    try:
        resp = requests.post(url, headers=headers, json=body, timeout=10)
        if resp.status_code == 200:
            lang = resp.json()["results"]["documents"][0]["detectedLanguage"]["name"]
            print(f"[PASS] Azure AI Services connected! Detected language: {lang}")
        else:
            print(f"[FAIL] Azure AI Services returned status {resp.status_code}")
            print(f"       Response: {resp.text[:200]}")
            print("       Check that your AZURE_AI_ENDPOINT and AZURE_AI_KEY are correct.")
    except Exception as e:
        print(f"[FAIL] Could not connect to Azure AI Services: {e}")
else:
    print("[SKIP] Azure AI Services not configured (AZURE_AI_ENDPOINT / AZURE_AI_KEY missing)")

## Step 4: Test Azure OpenAI

This sends a tiny chat request to your GPT-4o-mini deployment.

In [ ]:
from openai import AzureOpenAI

oai_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
oai_key = os.environ.get("AZURE_OPENAI_KEY", "")
oai_deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

if oai_endpoint and oai_key and "YOUR" not in oai_endpoint:
    try:
        client = AzureOpenAI(
            azure_endpoint=oai_endpoint,
            api_key=oai_key,
            api_version="2024-06-01"
        )
        response = client.chat.completions.create(
            model=oai_deployment,
            messages=[{"role": "user", "content": "Say 'Connection successful!' in exactly 2 words."}],
            max_tokens=10
        )
        reply = response.choices[0].message.content.strip()
        print(f"[PASS] Azure OpenAI connected! Model replied: {reply}")
    except Exception as e:
        print(f"[FAIL] Azure OpenAI error: {e}")
        print("       Check AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_KEY, and AZURE_OPENAI_DEPLOYMENT.")
else:
    print("[SKIP] Azure OpenAI not configured (AZURE_OPENAI_ENDPOINT / AZURE_OPENAI_KEY missing)")

## Step 5: Test Azure Speech

This verifies your Speech service key and region.

In [ ]:
speech_key = os.environ.get("AZURE_SPEECH_KEY", "")
speech_region = os.environ.get("AZURE_SPEECH_REGION", "")

if speech_key and speech_region:
    try:
        # Quick token fetch to verify credentials
        token_url = f"https://{speech_region}.api.cognitive.microsoft.com/sts/v1.0/issueToken"
        resp = requests.post(
            token_url,
            headers={"Ocp-Apim-Subscription-Key": speech_key},
            timeout=10
        )
        if resp.status_code == 200:
            print(f"[PASS] Azure Speech connected! Region: {speech_region}")
        else:
            print(f"[FAIL] Azure Speech returned status {resp.status_code}")
            print("       Check AZURE_SPEECH_KEY and AZURE_SPEECH_REGION in your .env file.")
    except Exception as e:
        print(f"[FAIL] Could not connect to Azure Speech: {e}")
else:
    print("[SKIP] Azure Speech not configured (AZURE_SPEECH_KEY / AZURE_SPEECH_REGION missing)")

## Step 6: Test Azure Document Intelligence

This verifies your Document Intelligence endpoint and key.

In [ ]:
doc_endpoint = os.environ.get("AZURE_DOC_INTEL_ENDPOINT", "").rstrip("/")
doc_key = os.environ.get("AZURE_DOC_INTEL_KEY", "")

if doc_endpoint and doc_key and "YOUR" not in doc_endpoint:
    try:
        # List available models to verify connectivity
        url = f"{doc_endpoint}/documentintelligence/documentModels?api-version=2024-02-29-preview"
        resp = requests.get(
            url,
            headers={"Ocp-Apim-Subscription-Key": doc_key},
            timeout=10
        )
        if resp.status_code == 200:
            models = resp.json().get("value", [])
            print(f"[PASS] Azure Document Intelligence connected! {len(models)} models available.")
        else:
            print(f"[FAIL] Document Intelligence returned status {resp.status_code}")
            print("       Check AZURE_DOC_INTEL_ENDPOINT and AZURE_DOC_INTEL_KEY.")
    except Exception as e:
        print(f"[FAIL] Could not connect to Document Intelligence: {e}")
else:
    print("[SKIP] Document Intelligence not configured (AZURE_DOC_INTEL_ENDPOINT / AZURE_DOC_INTEL_KEY missing)")

## Summary

If all tests above show `[PASS]`, you're ready to go!

**Next:** Open `01_responsible_ai.ipynb` and click **Run All**.

If any test shows `[FAIL]` or `[SKIP]`:
1. Open your `.env` file
2. Check the variable names and values
3. Refer to [docs/01_azure_portal_setup.md](../docs/01_azure_portal_setup.md)
4. See [docs/troubleshooting.md](../docs/troubleshooting.md) for common issues